In [4]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [5]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [11]:
data = pd.read_csv('../../data/ham_data.csv')

In [12]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [14]:
data[data["yesil_skor_notu"].isna()].sample(10)

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
34020,https://world.openfoodfacts.org/product/400040...,4.000405e+12,Vegane Mühlen Crispies Brezel – Rügenwalder Mü...,180.0,NaN,rügenwalder mühle,"Plant-based foods and beverages, ,, Plant-base...","Vegetarian, ,, Vegan, ,, European Vegetarian U...",NaN,NaN,...,0.0,1.0,7.0,NaN,3.0,0.0,Vegane Mühlen Crispies Brezel,g,"[Plant-based foods and beverages, Plant-based ...","[Vegetarian, Vegan, European Vegetarian Union,..."
14989,https://world.openfoodfacts.org/product/309007...,3.090070e+12,Chapelure Flocons de maïs et Aromates – Bloch ...,140.0,fr:Etui en carton,bloch,"Plant-based foods and beverages, ,, Plant-base...","Source of fibre, ,, High fibres, ,, Nutriscore...",NaN,"France, ,, Liverdun",...,0.0,NaN,0.0,3.0,2.0,0.0,Chapelure Flocons de maïs et Aromates,g,"[Plant-based foods and beverages, Plant-based ...","[Source of fibre, High fibres, Nutriscore, Nut..."
60886,https://world.openfoodfacts.org/product/400235...,4.002359e+12,SAUCE NEMS – SUZI WAN – 135 ml,135.0,NaN,suzi wan,"Condiments, ,, Sauces, ,, fr:Sauces pour nems,...",NaN,NaN,NaN,...,13.0,0.0,20.0,NaN,0.0,0.0,SAUCE NEMS,ml,"[Condiments, Sauces, fr:Sauces pour nems, fr:S...",[]
11976,https://world.openfoodfacts.org/product/541118...,5.411188e+12,Tempting and Tropical Coco – alpro – 1l,1.0,"Composite material, ,, Cardboard, ,, Tetra Pak...",alpro,"Beverages and beverages preparations, ,, Plant...","Low or no fat, ,, Vegetarian, ,, Low fat, ,, N...",Origine de la noix de coco : Indonésie / Phili...,France,...,0.0,1.0,0.0,0.0,0.0,0.0,Tempting and Tropical Coco,l,"[Beverages and beverages preparations, Plant-b...","[Low or no fat, Vegetarian, Low fat, No gluten..."
49831,https://world.openfoodfacts.org/product/600377...,6.003770e+12,Nandos Medium Marinade 262g – 262g,262.0,"Glass-bottle, ,, Steel-lid",nandos,"Condiments, ,, Groceries",NaN,NaN,NaN,...,1.0,0.0,11.0,NaN,0.0,0.0,Nandos Medium Marinade 262g,g,"[Condiments, Groceries]",[]
8175,https://world.openfoodfacts.org/product/376008...,3.760087e+12,Lait d'or – ETHNOSCIENCE – 150 g,150.0,NaN,ethnoscience,fr:Alimentation>Epicerie sucrée>Thés et tisanes,"Fair trade, ,, Organic, ,, EU Organic, ,, FR-B...",Inde (l'),NaN,...,0.0,0.0,0.0,3.0,5.0,0.0,Lait d'or,g,[fr:Alimentation>Epicerie sucrée>Thés et tisanes],"[Fair trade, Organic, EU Organic, FR-BIO-01, A..."
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2.0,NaN,sidi ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Sidi Ali,l,"[Beverages and beverages preparations, Beverag...",[Green Dot]
44267,https://world.openfoodfacts.org/product/405648...,5.000170e+12,Buttermilk pancakes,NaN,NaN,NaN,"Crêpes and galettes, ,, Pancakes",NaN,NaN,NaN,...,2.0,0.0,4.0,2.0,0.0,0.0,Buttermilk pancakes,NaN,"[Crêpes and galettes, Pancakes]",[]
54429,https://world.openfoodfacts.org/product/325039...,3.250392e+12,Boulettes au boeuf aux oignons 750g – Jean Roz...,750.0,NaN,jean rozé,"Meats and their products, ,, Beef and its prod...","French meat, ,, French beef, ,, Nutriscore",NaN,NaN,...,0.0,5.0,5.0,NaN,1.0,0.0,Boulettes au boeuf aux oignons 750g,g,"[Meats and their products, Beef and its produc...","[French meat, French beef, Nutriscore]"
64281,https://world.openfoodfacts.org/product/356470...,3.564701e+12,Bresaola italienne 7 tranches – Marque Repère ...,70.0,NaN,marque repère,"Meats and their products, ,, Beef and its prod...",NaN,NaN,NaN,...,0.0,1.0,17.0,NaN,0.0,0.0,Bresaola italienne 7 tranches,g,"[Meats and their products, Beef and its produc...",[]


In [15]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [16]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001434
urun_adi                         0.002868
miktar                          19.241029
ambalaj                         58.770043
markalar                         3.513754
kategoriler                      0.002868
etiketler                       30.545277
mensei                          76.027594
uretim_yerleri                  80.076299
satildigi_ulkeler                0.108998
icerik_metni                    12.884720
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   13.198807
nutriscore_notu                  0.103261
nova_grubu                      15.433267
yesil_skor_notu                 27.038694
palmiye_yagi_icermez            19.154978
vejetaryen                      22.860913
vegan_durumu                    12.669592
yag_seviyesi                     2.331985
doymus_yag_seviyesi              3.355993
seker_seviyesi                   2

In [17]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","etiketler","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)


In [18]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [19]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

TypeError: '<' not supported between instances of 'float' and 'str'

In [ ]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     39877
False    12897
Name: count, dtype: int64

In [ ]:
data.isnull().mean() * 100

barkod                         0.001546
miktar                        18.833027
markalar                       3.478032
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 12.517205
nutriscore_notu                0.091242
nova_grubu                    14.456490
yesil_skor_notu               25.758162
palmiye_yagi_icermez          18.386094
vejetaryen                    22.184248
vegan_durumu                  12.006866
enerji_kcal                    0.947992
yag_g                          0.954178
doymus_yag_g                   1.998051
karbonhidrat_g                 1.036141
seker_g                        1.408843
lif_g                         29.135673
protein_g                      0.960364
tuz_g                          0.759321
alkol_yuzde                   95.046626
meyve_sebze_baklagil_yuzde    71.070009
birim                         20.784684
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [ ]:
data.to_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/data.csv', index=False)

In [ ]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'alkol_yuzde', 'meyve_sebze_baklagil_yuzde',
       'birim', 'kategori_listesi', 'etiketler_listesi'],
      dtype='object')